In [1]:
import pandas as pd
import yaml
import importlib
import etl.normalization as normalization

importlib.reload(normalization)


<module 'etl.normalization' from 'c:\\Users\\dergun\\Documents\\hfqa_tool\\etl\\normalization.py'>

In [2]:
with open(r"C:\Users\dergun\Documents\hfqa_tool\hf_schema.yaml", "r", encoding="utf-8") as f:
    schema = yaml.safe_load(f)

schema

{'version': '1.0',
 'schema_id': 'heatflow:v1',
 'name': 'heat_flow_db_complete',
 'normalization': {'string': {'trim': True,
   'collapse_space': True,
   'case_insensitive': True,
   'enforce_brackets': True,
   'normalize_separator': True,
   'missing_tokens': ['', '-', 'NA', '<NA>', 'none', 'null']},
  'numeric': {'decimal_comma_to_dot': True,
   'strip_thousands_separators': [' ', '.', '_'],
   'missing_tokens': ['', 'NA', '<NA>', '-']}},
 'core': {'ID': {'dtype': 'int64', 'unique': True, 'min': 1},
  'Obligation': {'dtype': 'string', 'allowed': ['M', 'R', 'O', '-']},
  'Level': {'dtype': 'string', 'allowed': ['Parent', 'Child', 'Admin']}},
 'columns': {'P1': {'dtype': 'float64',
   'range': [-999999.9, 999999.9],
   'obligation': 'M',
   'comment': 'HF Value'},
  'P2': {'dtype': 'float64',
   'range': [0.0, 999999.9],
   'obligation': 'M',
   'comment': 'HF Uncertainty'},
  'P3': {'dtype': 'string', 'obligation': 'M', 'comment': 'Name'},
  'P4': {'dtype': 'float64',
   'range': [

In [3]:
df_raw = pd.read_excel(
    r"C:\Users\dergun\Documents\hfqa_tool\testing\Abc_xyz_2008.xlsx",
    sheet_name=1,
    header=0,
    dtype=str 
)
print(df_raw.shape)
df_raw.head(10)

#check if everything is string 
df_raw.dtypes

(28, 71)


c:\Users\dergun\Documents\hfqa_tool\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


ID    object
P1    object
P2    object
P3    object
P4    object
       ...  
A4    object
A5    object
A6    object
A7    object
A8    object
Length: 71, dtype: object

In [4]:
# Tranfrom Excel to Parquet (all to string)
#divide into meta and data
df_raw["row_type"] = ["meta"] * 7 + ["data"] * (len(df_raw) - 7)
df_raw_str = df_raw.astype("string[pyarrow]")
#convert to parquet
df_raw_str.to_parquet(
    r"C:\Users\dergun\Documents\hfqa_tool\testing\Abc_xyz_2008_raw.parquet",
    index=False)



In [5]:
# Load Parquet in as String 
df_raw_parquet_string = pd.read_parquet(
    r"C:\Users\dergun\Documents\hfqa_tool\testing\Abc_xyz_2008_raw.parquet",
    dtype_backend="pyarrow"
).astype("string[pyarrow]")  
df_raw_parquet_string 

df_raw_parquet_string.dtypes

ID          string[pyarrow]
P1          string[pyarrow]
P2          string[pyarrow]
P3          string[pyarrow]
P4          string[pyarrow]
                 ...       
A5          string[pyarrow]
A6          string[pyarrow]
A7          string[pyarrow]
A8          string[pyarrow]
row_type    string[pyarrow]
Length: 72, dtype: object

In [6]:
normalize_dataframe = normalization.normalize_only_data_rows(df_raw_parquet_string,schema)


In [7]:
normalize_dataframe

,ID,P1,P2,P3,P4,P5,P6,P7,P8,P9,...,C49,A1,A2,A3,A4,A5,A6,A7,A8,row_type
0,Obligation,M,M,M,M,M,M,M,R,R,...,O,-,-,-,-,-,-,-,-,meta
1,Domain,"B,S","B,S","B,S","B,S","B,S","B,S","B,S","B,S","B,S",...,"B,S",-,-,-,-,-,-,-,-,meta
2,Quality Relevance,U score,U score,-,-,-,M score,-,-,-,...,-,-,-,-,-,-,-,-,-,meta
3,Name,Heat-flow value,Heat-flow uncertainty,Site name,Geographical latitude,Geographical longitude,Elevation (Geographical),Basic geographical environment,General comments parent level,Flag heat production of the overburden (heat-f...,...,IGSN,Reviewer_name,Reviewer_comment,Review Date,Country,Region,Continent,Domain,Unique entry ID,meta
4,Short Name,q,q_uncertainty,name,lat_NS,long_EW,elevation,environment,p_comment,corr_HP_flag,...,Ref_ISGN,Reviewer_name,Reviewer_comment,Review_date,Country,Region,Continent,Domain,ID,meta
5,Unit,mW/m²,mW/m²,-,degrees,degrees,m,-,-,-,...,-,-,-,-,-,-,-,-,-,meta
6,Allowed range of values,"-999,999.9 – 999,999.9","0 – 999,999.9",-,-90.00000 – +90.00000,-180.00000 – +180.00000,-12000 – +9000,[Onshore (continental)],-,Yes,...,-,-,-,-,-,-,-,-,-,meta
7,1,3935592,33494400000000004,ch icheng 13,4068333333,1155,1300,(onshore (continental)),<NA>,[no],...,<NA>,Xiaoxue Jiang,Chicheng,20.09.2024,China,Hebei chicheng,Asia,continental,<NA>,data
8,2,33494400000000006,<NA>,fanshan 103,402,11543333333333334,750,[onshore (continental)],<NA>,[unspecified],...,<NA>,Xiaoxue Jiang,Fanshan,20.09.2024,China,Hebei chicheng,Asia,continental,<NA>,data
9,3,264,065,fangshan 46,4016666666,11543333333333334,875,[offshore (continental)],mean hf,[unspecified],...,<NA>,Xiaoxue Jiang,Fanshan,20.09.2024,China,Hebei fanshan,Asia,continental,<NA>,data
